# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ma5029blp-wq/ML-flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd
import numpy as np

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [4]:
# Check the distribution of content staleness

print("Days since last update:")
print(df["days_since_last_update"].describe())

print("\nMissing values:")
print(df["days_since_last_update"].isna().sum())

Days since last update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Missing values:
0


In [5]:
print("\nImpressions - last 30 days:")
print(df["impressions_last_30d"].describe())

print("\nImpressions - previous 30 days:")
print(df["impressions_prev_30d"].describe())


Impressions - last 30 days:
count     30000.000000
mean       1429.058733
std        5643.852081
min           0.000000
25%          10.000000
50%         139.000000
75%         768.000000
max      238796.000000
Name: impressions_last_30d, dtype: float64

Impressions - previous 30 days:
count     30000.000000
mean       1783.078500
std        6150.429511
min           0.000000
25%          19.000000
50%         210.000000
75%        1143.000000
max      218786.000000
Name: impressions_prev_30d, dtype: float64


In [6]:
# Calculate the observed change in impressions
# from the previous 30 days to the last 30 days.

df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    ((df["impressions_last_30d"] - df["impressions_prev_30d"])
     / df["impressions_prev_30d"]) * 100,
    np.nan
)

print(df["impression_change_pct"].describe())

count    26612.000000
mean        -4.785741
std        473.861561
min       -100.000000
25%        -62.632201
50%        -33.464585
75%          0.000000
max      44900.000000
Name: impression_change_pct, dtype: float64


In [7]:
df["declining_observed"] = (
    df["impressions_last_30d"] < df["impressions_prev_30d"]
).astype(int)

print(df["declining_observed"].value_counts())
print("\nDecline rate:", df["declining_observed"].mean())

declining_observed
1    19716
0    10284
Name: count, dtype: int64

Decline rate: 0.6572


In [8]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, np.inf],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("declining_observed", "mean")
      )
      .reset_index()
)

staleness_check["declining_rate_pct"] = (
    staleness_check["declining_rate"] * 100
).round(1)

print(staleness_check)

  staleness_bucket      n  declining_rate  declining_rate_pct
0        0-30 days  20480        0.614209                61.4
1       31-90 days    175        0.657143                65.7
2      91-180 days   9171        0.755861                75.6
3        181+ days    174        0.517241                51.7


In [9]:
print(df["impression_change_pct"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
))

count    26612.000000
mean        -4.785741
std        473.861561
min       -100.000000
10%        -85.350183
25%        -62.632201
50%        -33.464585
75%          0.000000
90%         50.000000
max      44900.000000
Name: impression_change_pct, dtype: float64


In [10]:
print(df["avg_position"].describe())

print("\nNo-data positions:")
print((df["avg_position"] == 0).sum())

print("\nPosition tiers:")
print(df["position_tier"].value_counts(dropna=False))

count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64

No-data positions:
1205

Position tiers:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64


In [12]:
# Check whether search visibility is associated with observed impression decline.

visibility_check = (
    df.groupby("impression_tier", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("declining_observed", "mean")
      )
      .reset_index()
)

visibility_check["declining_rate_pct"] = (
    visibility_check["declining_rate"] * 100
).round(1)

print(visibility_check)

  impression_tier      n  declining_rate  declining_rate_pct
0       excellent   1078        0.730983                73.1
1            good   7205        0.768217                76.8
2             low  11248        0.496799                49.7
3        moderate  10469        0.745534                74.6


In [13]:
visible = df["impression_tier"].isin(["moderate", "good", "excellent"])
stale = df["days_since_last_update"] >= 91

print("Meaningfully visible:", visible.sum())
print("Stale (91+ days):", stale.sum())
print("Both conditions:", (visible & stale).sum())
print("Percentage selected:", round((visible & stale).mean() * 100, 2), "%")

Meaningfully visible: 18752
Stale (91+ days): 9345
Both conditions: 7234
Percentage selected: 24.11 %


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** Prioritize content for refresh when it has meaningful search visibility** (moderate, good, or excellent)** and has not been updated for at least 91 days. Visibility is the main opportunity signal, while staleness is supporting evidence.

**Reason code: VISIBLE_STALE **— the page has meaningful search visibility and has not been updated recently.

**Action:** REFRESH.

**Signal verdicts:**

***Staleness: MIXED —*** decline rates increased through the 91–180 day bucket but dropped for 181+ days, so staleness did not show a consistent relationship.

***Impression visibility: CONFIRMED*** — moderate, good, and excellent impression tiers had substantially higher observed decline rates than the low-impression tier.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the transparent baseline score

import os

os.makedirs("work/outputs", exist_ok=True)

visibility_score = {
    "low": 0,
    "moderate": 1,
    "good": 2,
    "excellent": 3
}

df["visibility_score"] = df["impression_tier"].map(visibility_score).fillna(0)

df["stale_bonus"] = (
    (df["days_since_last_update"] >= 91).astype(int)
)

df["score"] = df["visibility_score"] + df["stale_bonus"]

# Action and reason code
df["action"] = np.where(
    (df["visibility_score"] >= 1) &
    (df["stale_bonus"] == 1),
    "REFRESH",
    "NO_ACTION"
)

df["reason_code"] = np.where(
    df["action"] == "REFRESH",
    "VISIBLE_STALE",
    "NONE"
)

# Rank highest-priority pages first
queue = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).copy()

queue["rank"] = np.arange(1, len(queue) + 1)

# Save the required output
output_cols = [
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
    "impression_tier",
    "impressions_90d",
    "days_since_last_update"
]

queue[output_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Queue shape:", queue.shape)
print("\nAction counts:")
print(queue["action"].value_counts())

print("\nTop 10:")
print(queue[output_cols].head(10))

Queue shape: (30000, 53)

Action counts:
action
NO_ACTION    22766
REFRESH       7234
Name: count, dtype: int64

Top 10:
       rank            content_id  score   action    reason_code  \
6653      1  content_5fe46e04994d      4  REFRESH  VISIBLE_STALE   
29400     2  content_2dba2b1f9536      4  REFRESH  VISIBLE_STALE   
13537     3  content_2c2606c5d176      4  REFRESH  VISIBLE_STALE   
26531     4  content_cb112fce36be      4  REFRESH  VISIBLE_STALE   
21565     5  content_9532f197bbc8      4  REFRESH  VISIBLE_STALE   
3394      6  content_36ff89c8214e      4  REFRESH  VISIBLE_STALE   
26798     7  content_b28d1efd668f      4  REFRESH  VISIBLE_STALE   
23767     8  content_813e88069237      4  REFRESH  VISIBLE_STALE   
26255     9  content_c21024970297      4  REFRESH  VISIBLE_STALE   
7445     10  content_c8e9d6ab9013      4  REFRESH  VISIBLE_STALE   

      impression_tier  impressions_90d  days_since_last_update  
6653        excellent           517715                     104  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 pages selected by the baseline rule.

For each page:
- **Action** shows what the baseline recommends.
- **Reason code** explains why the page was selected.
- **Confidence note** explains how strong the evidence looks.
- **What would make it wrong** identifies a situation where the refresh recommendation may not be appropriate.

The baseline selects pages with meaningful search visibility and at least 91 days since their last update. The review is a skeptical check of whether these recommendations make sense.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
review_cols = [
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
    "impression_tier",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "ctr",
    "content_type"
]

top20 = queue[review_cols].head(20).copy()

print(top20.to_string(index=False))

 rank           content_id  score  action   reason_code impression_tier  impressions_90d  impressions_last_30d  impressions_prev_30d  days_since_last_update  content_age_days  avg_position  ctr    content_type
    1 content_5fe46e04994d      4 REFRESH VISIBLE_STALE       excellent           517715                120791                218786                     104               537           4.2 0.14 keyword article
    2 content_2dba2b1f9536      4 REFRESH VISIBLE_STALE       excellent           443434                139891                137909                     104               299          27.9 0.21 keyword article
    3 content_2c2606c5d176      4 REFRESH VISIBLE_STALE       excellent           347399                104248                164079                     104               362           4.2 0.53 keyword article
    4 content_cb112fce36be      4 REFRESH VISIBLE_STALE       excellent           309910                 72468                124500                     104    

| Rank | Action | Reason code | Confidence note | What would make it wrong |
|---:|---|---|---|---|
| 1 | REFRESH | VISIBLE_STALE | High visibility and 104 days since update; impressions also declined substantially. | The traffic decline could be temporary or caused by something other than stale content. |
| 2 | REFRESH | VISIBLE_STALE | Very high visibility and 104 days since update; recent impressions are roughly stable. | The page may still be performing well and may not need a refresh. |
| 3 | REFRESH | VISIBLE_STALE | Excellent visibility, 104 days stale, and impressions declined. | The decline may be temporary or unrelated to content freshness. |
| 4 | REFRESH | VISIBLE_STALE | Excellent visibility and a large recent impression decline. | Search changes or competition could explain the decline instead. |
| 5 | REFRESH | VISIBLE_STALE | Excellent visibility and a substantial recent impression decline. | The decline may have another cause besides content freshness. |
| 6 | REFRESH | VISIBLE_STALE | Strong visibility and 104 days since update, but impressions are almost unchanged. | Stable performance may mean there is no urgent refresh opportunity. |
| 7 | REFRESH | VISIBLE_STALE | Excellent visibility and a recent impression decline. | The decline could be normal variation rather than a content problem. |
| 8 | REFRESH | VISIBLE_STALE | Excellent visibility and a noticeable recent impression decline. | The traffic loss could be unrelated to content freshness. |
| 9 | REFRESH | VISIBLE_STALE | Excellent visibility and a recent impression decline. | The decline may be too small to justify a refresh. |
| 10 | REFRESH | VISIBLE_STALE | Excellent visibility and a substantial recent impression decline. | The decline could have a cause unrelated to stale content. |
| 11 | REFRESH | VISIBLE_STALE | Excellent visibility and a recent impression decline. | The page may still be useful and accurate despite the decline. |
| 12 | REFRESH | VISIBLE_STALE | Excellent visibility and a recent impression decline. | The decline may not be caused by content staleness. |
| 13 | REFRESH | VISIBLE_STALE | Excellent visibility and a recent impression decline. | A moderate decline does not prove that refreshing will help. |
| 14 | REFRESH | VISIBLE_STALE | Excellent visibility and a slight recent impression decline. | The change may be normal fluctuation. |
| 15 | REFRESH | VISIBLE_STALE | Excellent visibility and a large recent impression decline. | External search or market changes could explain the decline. |
| 16 | REFRESH | VISIBLE_STALE | Excellent visibility and a recent impression decline. | The page may still be performing adequately. |
| 17 | REFRESH | VISIBLE_STALE | Excellent visibility and 104 days since update, but impressions are nearly stable. | Stable performance may mean the page does not need refreshing. |
| 18 | REFRESH | VISIBLE_STALE | Excellent visibility and 104 days since update. | **Weak pick: impressions increased strongly from 50,823 to 88,590, so the page is currently improving.** |
| 19 | REFRESH | VISIBLE_STALE | Excellent visibility and a substantial recent impression decline. | The decline may be temporary or caused by another factor. |
| 20 | REFRESH | VISIBLE_STALE | Excellent visibility and a slight recent impression decline. | The small decline may not justify the cost of a refresh. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The clearest weak pick is rank 18. It was given a REFRESH action because it has excellent visibility and has not been updated for 104 days, but its impressions increased from 50,823 in the previous 30 days to 88,590 in the last 30 days. This shows that the baseline can prioritize a page that is currently improving.

Ranks 2, 6, and 17 are also weaker picks because their recent impressions are approximately stable.

The baseline score uses only impression_tier and days_since_last_update. It does not use trend_direction, trend_pct, or is_declining_label. It also does not use a future window; the score is based on the available snapshot fields.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check which columns are actually used to build the baseline score

score_inputs = [
    "impression_tier",
    "days_since_last_update"
]

for col in score_inputs:
    print(f"Used in score: {col}")

forbidden_inputs = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nLeakage check:")
for col in forbidden_inputs:
    print(f"{col}: {'NOT USED' if col not in score_inputs else 'USED'}")

print("\nBaseline score formula:")
print("score = visibility_score + stale_bonus")
print("visibility_score comes from impression_tier")
print("stale_bonus = 1 when days_since_last_update >= 91")

Used in score: impression_tier
Used in score: days_since_last_update

Leakage check:
trend_direction: NOT USED
trend_pct: NOT USED
is_declining_label: NOT USED

Baseline score formula:
score = visibility_score + stale_bonus
visibility_score comes from impression_tier
stale_bonus = 1 when days_since_last_update >= 91


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.